# 00 — EDA of the Match Modeling Dataset

## Objective
This notebook performs an initial exploratory analysis of the processed international football match data used in the World Cup forecasting project.

## What is reviewed
- Team-level match feature dataset coverage
- Team and tournament frequency distributions
- Temporal coverage of the dataset
- Filtered modeling dataset sanity checks
- Elo rating table inspection
- Manual validation of World Cup group-team availability

## Why this notebook matters
The goal is to validate that the processed data has enough breadth, temporal depth, and team coverage to support:
1. match outcome modeling,
2. Elo-based strength estimation,
3. tournament simulation inputs.

## 1. Load the processed team-match feature dataset

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
from src.utils.config import PROCESSED_DATA_DIR
import pandas as pd

df = pd.read_parquet(PROCESSED_DATA_DIR / "team_match_features.parquet")

print(df.shape)
df.head()

In [ ]:
df["team"].nunique()

In [ ]:
teams = sorted(df["team"].unique())

teams[:50]

In [ ]:
df["team"].value_counts().head(20)

In [ ]:
df["tournament"].value_counts().head(20)

## 2. Temporal distribution of matches

This section checks how the volume of observations evolves over time.  
It is useful for identifying:
- periods with sparse historical coverage,
- modern data density,
- whether the modeling dataset is biased toward recent years.

In [ ]:
df.groupby(df["date"].dt.year).size().plot(figsize=(12,5))

## 3. Sanity check on the filtered modeling dataset

The project also uses a filtered match-level dataset.  
This block verifies:
- final row count,
- number of unique teams,
- most frequent tournaments,
- consistency between processed sources.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils.config import PROCESSED_DATA_DIR

df = pd.read_parquet(PROCESSED_DATA_DIR / "matches_filtered.parquet")

print(df.shape)
print(df["team"].nunique())
print(df["tournament"].value_counts().head(20))
print(sorted(df["team"].unique())[:50])

In [ ]:
df = pd.read_parquet(PROCESSED_DATA_DIR / "matches_filtered.parquet")

print(df.shape)
print(df["team"].nunique())
print(df["tournament"].value_counts().head(10))

## 4. Elo ratings inspection

The Elo table is one of the key components of the project because it provides a compact proxy for team strength.

Here we inspect:
- the Elo history table,
- latest top-ranked national teams,
- the recent evolution of selected teams.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils.config import PROCESSED_DATA_DIR

elo = pd.read_parquet(PROCESSED_DATA_DIR / "team_elo_ratings.parquet")

print(elo.shape)
elo.head()

In [ ]:
elo.groupby("team")["elo_after"].last().sort_values(ascending=False).head(20)

In [ ]:
elo[elo["team"] == "Spain"][["date", "elo_after"]].tail(20)

In [ ]:
team = "Argentina"
team_elo = elo[elo["team"] == team].sort_values("date")

team_elo.plot(x="date", y="elo_after", figsize=(12, 5), title=f"{team} Elo Rating Over Time")

In [ ]:
elo.groupby("team")["elo_after"].last().sort_values(ascending=False).head(20)

## 5. Team naming QA

This quick manual check looks for naming inconsistencies that often break simulation pipelines, such as:
- `USA` vs `United States`
- alternate spellings,
- historical aliases,
- formatting mismatches.

This is especially important for downstream group configs and tournament simulation.

In [ ]:
[t for t in teams if "United" in t]
[t for t in teams if "Korea" in t]
[t for t in teams if "Morocco" in t]
[t for t in teams if "Netherlands" in t]
[t for t in teams if "Nigeria" in t]
[t for t in teams if "Peru" in t]
[t for t in teams if "Portugal" in t]
[t for t in teams if "Senegal" in t]
[t for t in teams if "Serbia" in t]
[t for t in teams if "Switzerland" in t]
[t for t in teams if "Turkey" in t]
[t for t in teams if "Uruguay" in t]
[t for t in teams if "USA" in t]

## 6. World Cup group configuration validation

This final block verifies whether all teams defined in the tournament group configuration are present in the processed feature dataset.

A clean result here is an important integration check between:
- `configs/world_cup_groups.json`
- processed data tables
- the simulation engine input layer

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().resolve().parents[0]

groups_path = PROJECT_ROOT / "configs" / "world_cup_groups.json"

with open(groups_path, "r", encoding="utf-8") as f:
    groups = json.load(f)

print(groups_path)

In [ ]:
available = set(teams)

missing = []

for group_name, group_teams in groups.items():
    for team in group_teams:
        if team not in available:
            missing.append((group_name, team))

missing

In [ ]:
for group, team in missing:
    print(group, "->", team)

## Summary of outputs and main takeaways

### Dataset coverage
The `team_match_features.parquet` table contains **98,142 rows and 24 columns**, which indicates a solid historical base for feature-driven match modeling.  
The broader processed dataset includes **333 unique teams**, showing that the raw feature store is intentionally wide and includes non-standard or non-FIFA entities in addition to national teams.

### Most frequent teams and tournaments
The most repeated teams in the broader feature dataset include **Sweden, England, Argentina, Brazil, Germany, and South Korea**, which is consistent with historically active international football nations.  
At tournament level, **Friendlies** dominate the data volume, followed by **FIFA World Cup qualification**, **UEFA Euro qualification**, and other continental qualification tournaments. This is expected in international football and gives the model a large number of training examples beyond major tournaments.

### Filtered modeling dataset
The filtered match dataset contains **62,959 rows and 24 columns** across **185 unique teams**.  
This suggests that the modeling pipeline already applies a sensible restriction to focus on cleaner and more relevant national-team data, reducing noise relative to the broader 333-team source.

### Elo table quality
The Elo history table contains **63,404 rows and 17 columns**, which is enough to support rolling team-strength retrieval for simulation.  
The latest observed top Elo teams are led by **Spain**, followed by **Argentina** and **France**, with **Portugal, England, Brazil, Netherlands, Senegal, Ecuador, and Germany** also appearing near the top. From a forecasting perspective, this ranking is coherent and provides confidence that the Elo update process is behaving reasonably.

### Recent team trend example
The Spain Elo snapshot shows that the team remained above **2100 Elo** during parts of 2025 and ends the extracted period at **2118.25**, reinforcing that Spain enters the simulation engine as one of the strongest teams in the current data cut.

### Integration QA for tournament simulation
The validation against `world_cup_groups.json` returns an empty `missing` list.  
That is an important engineering result: **all configured World Cup teams are available in the processed feature dataset**, so the tournament simulator can consume the group configuration without immediate naming or lookup failures.

### Overall conclusion
This EDA supports three practical conclusions:
1. the project has sufficient historical depth for probabilistic match modeling,
2. the filtered dataset is aligned with international football forecasting use cases,
3. the simulation pipeline input layer appears consistent with the current tournament configuration.

As a next step, the project can move safely from data validation into:
- feature inspection,
- model experimentation,
- tournament simulation analysis.